R Data Cleaning and Visualization - <10/18/26> - <johhun3213>

In [1]:
library(writexl)
library(readxl) #load the readxl library for excel files
library(dplyr) #load the dplyr for data manipulation and transformation
library(IRdisplay) #load the IRDisplay lib to help interact with jupyter notebooks

todays_date <- format(Sys.Date(), "%Y-%m-%d")
student_id <- "johhun3213"

#read excel file into df

file_path <- "C:/Users/david/OneDrive/Desktop/ECPI/SDC380/PA2.5/AirBnBSummary_v2.xlsx"
df <- read_excel(file_path)

duplicate_rows_before <- df[duplicated(df),]

display_markdown(paste("###", student_id, "-", todays_date))

display_markdown("### Duplicate Rows before cleaning")
display(duplicate_rows_before)

#make copy of df and remove dups

df_cleaned <- df
df_cleaned <- df_cleaned %>% distinct()

#find dup rows after cleaning
duplicate_rows_after <- df_cleaned[duplicated(df_cleaned),]

#display duplicate rows after cleaning
display_markdown("### Duplicate Rows after cleaning")
display(duplicate_rows_after)


Attaching package: 'dplyr'


The following objects are masked from 'package:stats':

    filter, lag


The following objects are masked from 'package:base':

    intersect, setdiff, setequal, union




### johhun3213 - 2026-10-18

### Duplicate Rows before cleaning

id,host_id,host_name,neighbourhood,room_type,price,minimum_nights,number_of_reviews,availability_365,<StudentID>
<dbl>,<dbl>,<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>
2.725210e+07,205256572,Stephanie,West Olentangy,Entire home/apt,296,1,164,167,<johhun3213>
2.864045e+07,181661236,Chris,West Olentangy,Entire home/apt,105,2,280,170,<johhun3213>
6.208150e+17,453019578,Devon,West Olentangy,Entire home/apt,99,15,3,65,<johhun3213>
6.209180e+17,453019578,Devon,West Olentangy,Entire home/apt,90,30,1,0,<johhun3213>


### Duplicate Rows after cleaning

id,host_id,host_name,neighbourhood,room_type,price,minimum_nights,number_of_reviews,availability_365,<StudentID>
<dbl>,<dbl>,<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>


In [2]:
count_before <- table(df$`room_type`)

#Manually identified mispelled countries and their correction
misspellings_correction <- c('Private room' = 'Private Room')

df_cleaned$`room_type` <- as.character(df_cleaned$`room_type`)

# Count how many entries will be replaced
replaced_count <- sum(df_cleaned$`room_type` %in% names(misspellings_correction))

# Do the replacement
df_cleaned$`room_type` <- ifelse(df_cleaned$`room_type` %in% names(misspellings_correction), misspellings_correction[df_cleaned$`room_type`], df_cleaned$`room_type`)

count_after <- table(df_cleaned$`room_type`)

display_markdown(paste("###", student_id, "-", todays_date))
display_markdown(paste("**", replaced_count, "entries were replaced.**"))

### johhun3213 - 2026-10-18

** 240 entries were replaced.**

In [3]:
#find the number of missing values in each column before filling
missing_values_before <- sapply(df, function(x) sum( is.na(x)))
#display header text
display_markdown(paste("###", student_id, "-", todays_date))

display_markdown("### Number of missing values in each column")
display (missing_values_before)

#fill missing values with the median valuee of each column
df_cleaned <- df_cleaned %>% mutate(across(everything(), ~ ifelse(is.na(.), median (., na.rm=TRUE), .)))

missing_values_after <- sapply(df_cleaned, function(x) sum(is.na(x)))

display_markdown("### Number of missing values in each column after filling with median")

display(missing_values_after)

### johhun3213 - 2026-10-18

### Number of missing values in each column

id           host_id         host_name     neighbourhood 
                0                 0                 0                 0 
        room_type             price    minimum_nights number_of_reviews 
                0                 4                 0                 0 
 availability_365       <StudentID> 
                5                 0

### Number of missing values in each column after filling with median

id           host_id         host_name     neighbourhood 
                0                 0                 0                 0 
        room_type             price    minimum_nights number_of_reviews 
                0                 0                 0                 0 
 availability_365       <StudentID> 
                0                 0

In [4]:
outliers <- list()  # stores the row numbers of the outliers in each column

numerical_columns <- names(df_cleaned)[sapply(df_cleaned, is.numeric)]
numerical_columns <- setdiff(numerical_columns, c("id", "host_id"))  # IDs aren't measurements, so skip them

# Find outliers using the 3.5 * IQR method
for (col in numerical_columns) {
    Q1 <- quantile(df_cleaned[[col]], 0.25, na.rm = TRUE)
    Q3 <- quantile(df_cleaned[[col]], 0.75, na.rm = TRUE)
    IQR <- Q3 - Q1
    outliers[[col]] <- which(df_cleaned[[col]] < Q1 - 3.5 * IQR | df_cleaned[[col]] > Q3 + 3.5 * IQR)
}

display_markdown(paste("###", student_id, "-", todays_date))

# Show how many outliers each column had
display_markdown("### Outliers removed")
for (col in names(outliers)) {
    display_markdown(paste0("**", col, ":** ", length(outliers[[col]]), " outliers"))
}

# Remove every row that was an outlier in any column
all_outliers <- unique(unlist(outliers))
rows_before <- nrow(df_cleaned)
if (length(all_outliers) > 0) {
    df_cleaned <- df_cleaned[-all_outliers, ]
}

display_markdown(paste0("**After cleaning: ", rows_before - nrow(df_cleaned),
                        " rows removed, ", nrow(df_cleaned), " rows remaining.**"))

### johhun3213 - 2026-10-18

### Outliers removed

**price:** 44 outliers

**minimum_nights:** 281 outliers

**number_of_reviews:** 60 outliers

**availability_365:** 0 outliers

**After cleaning: 378 rows removed, 1397 rows remaining.**

In [5]:
cleaned_file_path <- "C:/Users/david/OneDrive/Desktop/ECPI/SDC380/PA2.5/AirBnBSummary_v2_cleaned.xlsx"
write_xlsx(df_cleaned, cleaned_file_path)